# Choosing and configuring decoders

qLDPC distinguishes two kinds of decoders:

- An **error decoder** maps a syndrome to an inferred physical error, with a `decode_errors` method, or its alias `decode`.
  Decoder-based distance bounds and sliding-window decoders need inferred physical errors.
- An **observable decoder** maps a syndrome to predicted observable flips, with a `decode_observables` method.
  Circuit-level simulations use observable decoders. Code-capacity estimates accept either kind: they compare predicted and actual observable flips.

This notebook shows how to configure built-in decoders, how to supply custom decoders of your own, and how to predict observable flips in circuit-level simulations with Sinter.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import functools
import inspect
import itertools
import os

import numpy as np
import numpy.typing as npt
import sinter
import stim

from qldpc import codes, decoders
from qldpc.objects import Pauli

NUM_WORKERS = max(1, min(4, (os.cpu_count() or 1) - 1))

## Configure a built-in decoder

Each built-in decoder has a helper function in `qldpc.decoders`:
`bp_osd`, `bp_lsd`, `bf`, `mwpm`, `relay_bp`, `min_sum_bp`, `lookup`, `ilp`, and `guf`.
A helper returns a `DecoderSpec`, which only stores settings.
The method that consumes a `DecoderSpec` builds a decoder once it knows which parity check matrix or detector error model to decode, so one `DecoderSpec` can configure decoders for many matrices.

In [3]:
bp_lsd_settings = decoders.bp_lsd(max_iter=30, bp_method="minimum_sum")
bp_lsd_settings

decoders.bp_lsd(max_iter=30, bp_method='minimum_sum')

The signature of a helper lists the options of its decoder, together with their default values.
Autocomplete and `help(decoders.bp_lsd)` show the same information, with a description of each option.

In [4]:
for parameter in inspect.signature(decoders.bp_lsd).parameters.values():
    print(parameter)

error_channel: float | NDArray[numpy.floating] | collections.abc.Sequence[float] | None = None
max_iter: int = 0
bp_method: Literal['product_sum', 'minimum_sum', 'ps', 'ms'] = 'product_sum'
ms_scaling_factor: float = 1.0
schedule: Literal['parallel', 'serial'] = 'parallel'
omp_thread_count: int = 1
random_schedule_seed: int = 0
random_serial_schedule: bool = False
serial_schedule_order: collections.abc.Sequence[int] | None = None
bits_per_step: int = 1
lsd_method: Literal['LSD_0', 'LSD_E', 'LSD_CS'] = 'LSD_0'
lsd_order: int = 0
always_run_lsd: bool = False
error_rate: float | None = None


A misspelled option is rejected immediately, rather than silently ignored by the underlying decoder.
(The exception is `ilp`, which forwards additional options to the integer-program solver that it configures.)

In [5]:
try:
    decoders.bp_lsd(lsd_ordr=2)
except TypeError as error:
    print(f"TypeError: {error}")

TypeError: bp_lsd() got an unexpected keyword argument 'lsd_ordr'


## Decode syndromes

The `build` method of decoder settings builds a decoder for a parity check matrix or detector error model.

In [6]:
code = codes.HammingCode(3)
error = code.field.Zeros(len(code))
error[2] = 1
syndrome = code.matrix @ error

print("syndrome:", syndrome)
bp_lsd_decoder = bp_lsd_settings.build(code.matrix)
print("BP+LSD:", bp_lsd_decoder.decode_errors(syndrome))

bp_osd_decoder = decoders.bp_osd(osd_method="OSD_CS", osd_order=2).build(code.matrix)
print("BP+OSD:", bp_osd_decoder.decode_errors(syndrome))

syndrome: [0 1 1]
BP+LSD: [0 0 1 0 0 0 0]
BP+OSD: [0 0 1 0 0 0 0]


## Use decoders in code methods

Code methods accept the same `decoder` argument.
For example, the logical error rate of a surface code in a code-capacity model, decoded with minimum-weight perfect matching (MWPM):

In [7]:
surface_code = codes.SurfaceCode(3)
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000, max_error_rate=0.1, decoder=decoders.mwpm()
)
logical_error_rate(0.01)  # (logical error rate, uncertainty) at a physical error rate of 1%

(0.0014746379451472634, 8.386095725759955e-05)

A CSS code decodes X-type and Z-type errors independently, and each sector can use a different decoder:

- `decoder_x` infers X-type errors from their syndrome with respect to the Z-type stabilizers, and
- `decoder_z` infers Z-type errors from their syndrome with respect to the X-type stabilizers.

A sector that is not configured explicitly falls back to the shared `decoder` argument.

In [8]:
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000,
    max_error_rate=0.1,
    decoder_x=decoders.mwpm(),
    decoder_z=decoders.bp_osd(),
)
logical_error_rate(0.01)

(0.001559700539789863, 8.423686347434964e-05)

Randomized distance bounds also accept a decoder.
Providing one selects the decoder-based distance-bounding algorithm.

In [9]:
hgp_code = codes.HGPCode(codes.RepetitionCode(3))
hgp_code.get_code_params(bound=20, decoder=decoders.bp_osd(osd_method="OSD_CS", osd_order=2))

(13, 1, 3)

## Supply a custom decoder

A custom error decoder subclasses `decoders.ErrorDecoder`, and implements a `decode_errors` method that maps a syndrome to an inferred error.
The subclass inherits `decode` as an alias for `decode_errors`.
(Methods that use an error decoder also accept any object whose `decode` method returns an inferred error, such as a decoder from the `ldpc` package, and wrap it to provide `decode_errors`.)
As an example, the decoder below enumerates low-weight errors when it is built, and decodes each syndrome to a minimum-weight error that is consistent with it.
Its constructor accepts a parity check matrix or a detector error model, like the built-in decoders.

In [10]:
class MinimumWeightDecoder(decoders.ErrorDecoder):
    """Decode each syndrome to a minimum-weight error, found by enumerating low-weight errors.

    This decoder assumes binary (GF(2)) arithmetic, and is only practical for small decoding problems.
    """

    def __init__(
        self,
        pcm_or_dem: npt.NDArray[np.int_] | stim.DetectorErrorModel,
        max_weight: int = 2,
    ) -> None:
        if isinstance(pcm_or_dem, stim.DetectorErrorModel):
            matrix = decoders.DetectorErrorModelArrays(pcm_or_dem).detector_flip_matrix.toarray()
        else:
            matrix = np.asarray(pcm_or_dem).view(np.ndarray)
        self.num_errors = matrix.shape[1]

        # enumerate errors in order of increasing weight, keeping the first error for each syndrome
        self.syndrome_to_error: dict[tuple[int, ...], npt.NDArray[np.int_]] = {}
        for weight in range(max_weight + 1):
            for support in itertools.combinations(range(self.num_errors), weight):
                error = np.zeros(self.num_errors, dtype=int)
                error[list(support)] = 1
                syndrome = tuple((matrix @ error % 2).tolist())
                self.syndrome_to_error.setdefault(syndrome, error)

    def decode_errors(self, syndrome: npt.NDArray[np.int_]) -> npt.NDArray[np.int_]:
        """Decode a syndrome to a minimum-weight error, or to no error if none was found."""
        key = tuple(np.asarray(syndrome).view(np.ndarray).astype(int).tolist())
        no_error = np.zeros(self.num_errors, dtype=int)
        return self.syndrome_to_error.get(key, no_error).copy()

A custom decoder can be passed as `decoder=` in three ways, as shown below with the code-capacity logical error rate of the Hamming code:

1. As a **constructor**, such as the class itself, which qLDPC calls with the matrix or detector error model to decode.
2. As any other **callable** that builds a decoder from a matrix or detector error model, which is a convenient way to fix constructor options.
3. As a **prebuilt decoder**, which qLDPC uses as is.

A custom decoder can optionally define a `decode_errors_batch` method to decode many syndromes at once, or set `has_erasure_bit = True` to declare that it appends an erasure flag to each inferred error.
See the [decoder guide](https://qldpc.readthedocs.io/decoders.html) for details.

In [11]:
for label, decoder_input in [
    ("class", MinimumWeightDecoder),
    ("callable", functools.partial(MinimumWeightDecoder, max_weight=1)),
    ("prebuilt", MinimumWeightDecoder(code.matrix, max_weight=1)),
]:
    logical_error_rate = code.get_logical_error_rate_func(
        num_samples=2_000, max_error_rate=0.1, decoder=decoder_input
    )
    print(f"{label}:", logical_error_rate(0.01))

class: (0.002031041634940002, 3.277459255052654e-05)
callable: (0.002031041634940002, 3.277459255052654e-05)
prebuilt: (0.002031041634940002, 3.277459255052654e-05)


A prebuilt decoder only works for the matrix that it was built for.
The X-sector and Z-sector decoders of a CSS code decode different stabilizer matrices, so prebuilt decoders for a CSS code are passed per sector:

In [12]:
stabilizer_ops_x = surface_code.get_stabilizer_ops(Pauli.X)
stabilizer_ops_z = surface_code.get_stabilizer_ops(Pauli.Z)
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000,
    max_error_rate=0.1,
    decoder_x=MinimumWeightDecoder(stabilizer_ops_z),
    decoder_z=MinimumWeightDecoder(stabilizer_ops_x),
)
logical_error_rate(0.01)

(0.0014978623678756997, 8.394766862261419e-05)

Some methods decode a matrix that they construct internally.
Randomized distance bounds, for example, decode an effective check matrix that stacks the checks and logical operators of a code.
These methods reject prebuilt decoders, but accept a constructor, which they call with the matrix that they build:

In [13]:
try:
    hgp_code.get_distance_bound(decoder=MinimumWeightDecoder(hgp_code.matrix_z))
except ValueError as error:
    print(f"ValueError: {error}")

hgp_code.get_distance_bound(
    num_trials=5, decoder=functools.partial(MinimumWeightDecoder, max_weight=3)
)

ValueError: A prebuilt decoder cannot be passed as decoder= here because decoder-based distance bounds decode an internal effective check matrix that stacks the checks and logical operators of the code.  Pass decoder settings such as decoder=decoders.bp_osd(...), or a decoder constructor, instead


3

## Predict observable flips

The `build_observable_decoder` method of decoder settings builds an observable decoder for a detector error model.
The settings of MWPM, Relay-BP (`relay_bp` and `min_sum_bp`), and lookup-table decoders build decoders that predict observable flips natively.
For example, `lookup` builds an `ObservableLookupDecoder`, which maps each syndrome directly to its most likely observable flip.
The settings of other decoders build an error decoder, whose inferred errors are converted into observable flips.
`DecoderSpec.predicts_observables_natively` reports which of these applies.

Here, we compare a few decoders on a repetition-code memory experiment:

In [14]:
circuit = stim.Circuit.generated(
    "repetition_code:memory", distance=5, rounds=10, after_clifford_depolarization=0.01
)
dem = circuit.detector_error_model()
sampler = circuit.compile_detector_sampler(seed=0)
detection_events, observable_flips = sampler.sample(10_000, separate_observables=True)


def get_logical_error_rate(observable_decoder: decoders.ObservableDecoder) -> float:
    """The fraction of sampled shots whose observable flips a decoder predicts incorrectly."""
    predicted_flips = [observable_decoder.decode_observables(events) for events in detection_events]
    return float(np.mean(np.any(predicted_flips != observable_flips, axis=1)))


for settings in [
    decoders.mwpm(),
    decoders.relay_bp(),
    decoders.lookup(max_weight=2),
    decoders.bp_osd(),
]:
    observable_decoder = settings.build_observable_decoder(dem)
    print(
        f"{settings}: native={settings.predicts_observables_natively},"
        f" logical error rate {get_logical_error_rate(observable_decoder):.4f}"
    )

decoders.mwpm(): native=True, logical error rate 0.0003
decoders.relay_bp(): native=True, logical error rate 0.0007
decoders.lookup(max_weight=2): native=True, logical error rate 0.0047
decoders.bp_osd(): native=False, logical error rate 0.0004


A custom error decoder, such as `MinimumWeightDecoder`, can be converted in the same way with `decoders.ErrorsToObservablesDecoder`.
A custom **observable** decoder only needs a `decode_observables` method.
As an example, the decoder below predicts the observable flips of the minimum-weight errors that `MinimumWeightDecoder` infers, which gives the same predictions as converting `MinimumWeightDecoder` automatically:

In [15]:
class MinimumWeightObservableDecoder:
    """Predict the observable flips of a minimum-weight error that is consistent with a syndrome."""

    def __init__(self, dem: stim.DetectorErrorModel, max_weight: int = 2) -> None:
        self.error_decoder = MinimumWeightDecoder(dem, max_weight)
        self.observable_flip_matrix = decoders.DetectorErrorModelArrays(dem).observable_flip_matrix

    def decode_observables(self, syndrome: npt.NDArray[np.int_]) -> npt.NDArray[np.int_]:
        """Decode a syndrome, and return the observable flips of the inferred error."""
        return self.observable_flip_matrix @ self.error_decoder.decode_errors(syndrome) % 2


converted_decoder = decoders.ErrorsToObservablesDecoder(MinimumWeightDecoder(dem), dem)
custom_observable_decoder = MinimumWeightObservableDecoder(dem)
print(f"converted MinimumWeightDecoder: {get_logical_error_rate(converted_decoder):.4f}")
print(f"MinimumWeightObservableDecoder: {get_logical_error_rate(custom_observable_decoder):.4f}")

converted MinimumWeightDecoder: 0.0047
MinimumWeightObservableDecoder: 0.0047


Unlike the built-in decoders above, `MinimumWeightDecoder` ignores the error probabilities of the detector error model, and gives up on syndromes of errors with weight greater than 2, so it performs worse here.

## Simulate circuits with Sinter

A `SinterDecoder` wraps decoder settings, or a constructor of an error or observable decoder.
When Sinter compiles it for a detector error model, it builds an observable decoder for that model, natively if possible.
The compiled decoder predicts the observable flips of one shot with `decode_observables`, or of many shots with `decode_shots`.

Sinter decoders build a new decoder for each (simplified) detector error model, window, or subgraph that they decode, so they reject prebuilt decoders:

In [16]:
try:
    decoders.SinterDecoder(decoder=MinimumWeightDecoder(dem))
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: A prebuilt decoder cannot be passed as decoder= here because a SinterDecoder builds a new decoder for each (simplified) detector error model that it is compiled for.  Pass decoder settings such as decoder=decoders.bp_osd(...), or a decoder constructor, instead


Sinter decoders plug into `sinter.collect` as custom decoders.
Below, we compare decoding the whole memory experiment at once against decoding it in sliding windows of four rounds, which commit corrections two rounds at a time.
A sliding-window decoder commits the errors that it infers in each window, so it always uses an error decoder, even with settings that could predict observable flips natively.

Sinter samples in separate worker processes, which receive decoders by pickling them.
Decoder specs such as `decoders.mwpm()` pickle cleanly; to use a custom decoder with multiple workers, define its class in an importable module rather than in a notebook.

In [17]:
custom_decoders = {
    "monolithic": decoders.SinterDecoder(decoder=decoders.mwpm()),
    "sliding window": decoders.SlidingWindowDecoder(
        window_size=4, stride=2, decoder=decoders.mwpm()
    ),
}
stats = sinter.collect(
    tasks=[sinter.Task(circuit=circuit, decoder=name) for name in custom_decoders],
    custom_decoders=custom_decoders,
    num_workers=NUM_WORKERS,
    max_shots=20_000,
    max_errors=200,
    print_progress=False,
)
for stat in sorted(stats, key=lambda stat: stat.decoder):
    print(f"{stat.decoder}: logical error rate {stat.errors / stat.shots:.4f} ({stat.shots} shots)")

monolithic: logical error rate 0.0004 (20000 shots)
sliding window: logical error rate 0.0005 (20000 shots)


With this number of shots, each decoder makes only a handful of mistakes, so the difference between them is within statistical noise.
In general, a sliding-window decoder trades some accuracy for a bounded decoding latency; see the [sliding-window decoding example](https://qldpc.readthedocs.io/examples/logical_error_rates/4_sliding_window_decoding.html) for a more careful comparison.